In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
        )

        self.policy_head = nn.Linear(32, act_dim)
        self.value_head = nn.Linear(32, 1)

    def forward(self, x):
        x = self.shared(x)
        logits = self.policy_head(x)
        value = self.value_head(x)
        return logits, value

    def act(self, state):
        logits, value = self.forward(state)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)

        action = dist.sample()
        log_prob = dist.log_prob(action)

        return action, log_prob, value

    def evaluate(self, states, actions):
        logits, values = self.forward(states)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)

        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()

        return log_probs, values.squeeze(-1), entropy

In [2]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.dones = []
        self.values = []

    def clear(self):
        self.__init__()

    def compute_returns_advantages(self, gamma=0.99, lam=0.95, num_envs=4):
        T = len(self.rewards)
        rollout_steps = T // num_envs

        # --- reshape into (num_envs, rollout_steps) ---
        rewards = torch.tensor(self.rewards, dtype=torch.float32).view(num_envs, rollout_steps)
        dones = torch.tensor(self.dones, dtype=torch.float32).view(num_envs, rollout_steps)
        values = torch.tensor(self.values, dtype=torch.float32).view(num_envs, rollout_steps)

        returns = torch.zeros_like(rewards)
        advantages = torch.zeros_like(rewards)

        # --- compute GAE per environment ---
        for env in range(num_envs):
            gae = 0
            next_value = 0  

            for t in reversed(range(rollout_steps)):
                delta = (
                    rewards[env, t]
                    + gamma * next_value * (1 - dones[env, t])
                    - values[env, t]
                )

                gae = delta + gamma * lam * (1 - dones[env, t]) * gae

                advantages[env, t] = gae
                returns[env, t] = gae + values[env, t]

                next_value = values[env, t]

        # --- flatten back to original shape (T,) ---
        returns = returns.view(-1)
        advantages = advantages.view(-1)

        return returns, advantages

In [3]:
import soccer_twos
from gym_unity.envs import ActionFlattener


def make_env(worker_id,mode=None):
    if mode is None:
        env = soccer_twos.make(worker = worker_id,)
    if mode == "random":
        env = soccer_twos.make(
            variation=soccer_twos.EnvType.team_vs_policy,
            single_player=True,
            worker = worker_id,
            flatten_branched=True,
        )
    if mode == "still":
        env = soccer_twos.make(
            opponent_policy=lambda *_: 0, 
            variation=soccer_twos.EnvType.team_vs_policy,
            single_player=True,
            worker = worker_id,
            flatten_branched=True,
        )
    # print(env.action_space.nvec)
    return env

In [ ]:
import numpy as np
env = soccer_twos.make(
        variation=soccer_twos.EnvType.team_vs_policy,
        single_player=True,
        flatten_branched=True,
        worker = 2
    )
try: 
    print(env.action_space.n)
    print(env.observation_space.shape)
finally:env.close()
env = soccer_twos.make(
    )
try: 
    act = {i:np.array([0,0,0]) for i in range(4)}
    obs, reward, done, info = env.step(act)
    flattener = ActionFlattener(env.action_space.nvec)
    print(flattener.action_space.n)
    print(env.observation_space.shape)
    print(env.reset()[1].shape)
finally: env.close()

In [ ]:
env = make_env(3,"random")
try: 
    obs = env.reset()
    print("reset obs shape:", obs.shape)

    action = env.action_space.sample()
    print(f"action shape: {action}")
    obs, reward, done, info = env.step(0)

    print("step obs shape:", obs.shape)
    print("reward:", reward)
    print("done:", done)
    print(info)
finally:
    env.close()

In [4]:
from torch.utils.tensorboard import SummaryWriter
import time

log_dir = f"runs/soccer_ppo_{int(time.time())}"
writer = SummaryWriter(log_dir=log_dir)
%load_ext tensorboard


In [9]:
%tensorboard --logdir runs/

Reusing TensorBoard on port 6006 (pid 544673), started 2:57:40 ago. (Use '!kill 544673' to kill it.)

In [6]:
def info_reward(info,team_signs):
    player_info = info["player_info"]
    ball_info = info["ball_info"]

    player_pos = player_info["position"]
    player_vel = player_info["velocity"]
    ball_pos = ball_info["position"]
    ball_vel = ball_info["velocity"]

    team_sign = team_signs

    # --- compute ---
    to_ball = ball_pos - player_pos
    dist = np.linalg.norm(to_ball)
    to_ball_dir = to_ball / (dist + 1e-8)

    is_touch = dist < 1.5

    r_chase = 0.01 * np.dot(player_vel, to_ball_dir)
    r_dist = -0.002 * dist
    r_goal_dir = 0.02 * team_sign * ball_vel[0]

    r_impact = 0.0
    if is_touch:
        ball_speed = np.linalg.norm(ball_vel)
        r_impact = 0.02 * ball_speed + 0.05 * team_sign * ball_vel[0]

    # --- total reward ---
    return r_chase, r_dist, r_goal_dir, r_impact

In [7]:
import torch.optim as optim
import numpy as np
# hyperparameters
NUM_ENVS = 1
ROLLOUT_STEPS = 512
EPOCHS = 10
BATCH_SIZE = 256
GAMMA = 0.99
LAMBDA = 0.95
CLIP_EPS = 0.2
LR = 3e-4

episode_rewards = []
current_rewards = [0 for _ in range(NUM_ENVS)]
global_step = 0


def collect_rollout(envs,model):
    global global_step
    buffer = RolloutBuffer()

    states = []
    for i,env in enumerate(envs):
        states.append(env.reset()[:obs_dim])
    states = np.array(states)
    team_signs = [None for _ in range(NUM_ENVS)]
    for _ in range(ROLLOUT_STEPS):
        state_tensor = torch.tensor(states, dtype=torch.float32)

        with torch.no_grad():
            actions, log_probs, values = model.act(state_tensor)

        next_states = []
        rewards = []
        dones = []

        for i, env in enumerate(envs):
                
            action = actions[i].item()
            obs, reward, done, info = env.step(action)
            if team_signs[i] is None:
                player_x = info["player_info"]["position"][0]
                team_signs[i] = +1 if player_x < 0 else -1
            r_chase, r_dist, r_goal_dir, r_impact = info_reward(info,team_signs[i])
            r = reward + r_chase + r_dist + r_goal_dir + r_impact # modify here later
            writer.add_scalar("reward/env", reward, global_step)
            writer.add_scalar("reward/chase", r_chase, global_step)
            writer.add_scalar("reward/dist", r_dist, global_step)
            writer.add_scalar("reward/goal_dir", r_goal_dir, global_step)
            writer.add_scalar("reward/impact", r_impact, global_step)
            writer.add_scalar("reward/total", r, global_step)
            if done:
                next_obs = env.reset()
                team_signs[i] = None

            else:
                next_obs = obs

            next_states.append(next_obs[:obs_dim])
            rewards.append(r)
            dones.append(done)

        buffer.states.extend(state_tensor)
        buffer.actions.extend(actions)
        buffer.log_probs.extend(log_probs)
        buffer.rewards.extend(rewards)
        buffer.dones.extend(dones)
        buffer.values.extend(values.detach().view(-1).tolist())

        states = np.array(next_states)
        global_step += NUM_ENVS

    return buffer


def ppo_update(buffer,optimizer):
    returns, advantages = buffer.compute_returns_advantages(GAMMA, LAMBDA)

    states = torch.stack(buffer.states)
    actions = torch.stack(buffer.actions)
    old_log_probs = torch.stack(buffer.log_probs)

    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    for _ in range(EPOCHS):
        for i in range(0, len(states), BATCH_SIZE):
            s = states[i:i+BATCH_SIZE]
            a = actions[i:i+BATCH_SIZE]
            old_lp = old_log_probs[i:i+BATCH_SIZE]
            adv = advantages[i:i+BATCH_SIZE]
            ret = returns[i:i+BATCH_SIZE]

            log_probs, values, entropy = model.evaluate(s, a)

            ratio = torch.exp(log_probs - old_lp)
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv

            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = (ret - values).pow(2).mean()

            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            writer.add_scalar("Loss/actor", actor_loss.item(), global_step)
            writer.add_scalar("Loss/critic", critic_loss.item(), global_step)
            writer.add_scalar("Loss/total", loss.item(), global_step)
            writer.add_scalar("Stats/entropy", entropy.mean().item(), global_step)


    


In [ ]:
# training loop
# create environments
envs = [make_env(i+1,mode="still") for i in range(NUM_ENVS)]
obs_dim = envs[0].observation_space.shape[0]
act_dim = envs[0].action_space.n

model = ActorCritic(obs_dim, act_dim)
optimizer = optim.Adam(model.parameters(), lr=LR)
try:
    for episode in range(2000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_still_2000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()

In [ ]:
envs = [make_env(i+1,mode="random") for i in range(NUM_ENVS)]
try:
    for episode in range(2000,6000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_random_4000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
    writer.close()

In [8]:
envs = [make_env(i+1,mode="random") for i in range(NUM_ENVS)]
obs_dim = envs[0].observation_space.shape[0]
act_dim = envs[0].action_space.n
check_point_path = "ppo_checkpoint_random_4000.pth"
global_step = 6000

model = ActorCritic(obs_dim, act_dim)
optimizer = optim.Adam(model.parameters(), lr=LR)
model.load_state_dict(torch.load(check_point_path))
try:
    for episode in range(6000,10000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_random_8000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
    writer.close()

I0000 00:00:1776376139.482323  544624 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0
/home/clark/anaconda3/envs/soccertwos/lib/python3.8/site-packages/torch/autograd/__init__.py:145: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at  /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  Variable._execution_engine.run_backward(


Saved checkpoint at episode 6049
Saved checkpoint at episode 6099
Saved checkpoint at episode 6149
Saved checkpoint at episode 6199
Saved checkpoint at episode 6249
Saved checkpoint at episode 6299
Saved checkpoint at episode 6349
Saved checkpoint at episode 6399
Saved checkpoint at episode 6449
Saved checkpoint at episode 6499
Saved checkpoint at episode 6549
Saved checkpoint at episode 6599
Saved checkpoint at episode 6649
Saved checkpoint at episode 6699
Saved checkpoint at episode 6749
Saved checkpoint at episode 6799
Saved checkpoint at episode 6849
Saved checkpoint at episode 6899
Saved checkpoint at episode 6949
Saved checkpoint at episode 6999
Saved checkpoint at episode 7049
Saved checkpoint at episode 7099
Saved checkpoint at episode 7149
Saved checkpoint at episode 7199
Saved checkpoint at episode 7249
Saved checkpoint at episode 7299
Saved checkpoint at episode 7349
Saved checkpoint at episode 7399
Saved checkpoint at episode 7449
Saved checkpoint at episode 7499
Saved chec